<h1 style='font-weight:bold; color:#003366; text-align:center;'>Tema 2: Linting y calidad de código</h1>

<h3 style='font-weight:bold; color:#003366; text-align:center;'>Programación Analítica II</h3>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>Docente Camila Palomeque</h4>
<h4 style='font-weight:bold; color:#003366; text-align:center;'>S2 . 2026</h4>

## Objetivos del notebook

1. Refrescar la sintaxis de Python que vamos a usar todo el semestre: listas, diccionarios, bucles y f-strings.
2. Entender qué es el **análisis estático** y por qué encuentra errores sin ejecutar el código.
3. Usar **flake8** para detectar problemas de estilo y bugs reales, y leer su salida código por código.
4. Usar **black** para formatear automáticamente, sabiendo qué arregla y qué no.
5. Usar **pylint** para mirar el diseño del código y leer su puntaje sin obsesionarse con él.

<h2 style='font-weight:bold; color:#003366;'>Antes de arrancar: repaso de Python</h2>

- **variables**
- **listas**
- **diccionarios**
- **bucles con `if`**

**Formatos de f-string** 
- `{monto:,.2f}`: pone separador de miles y dos decimales (`1250000` se ve `1,250,000.00`)
- `{proporcion:.1%}`: muestra un número entre 0 y 1 como porcentaje (`0.333` se ve `33.3%`). 

Extra: en Python podés escribir `1_250_000` en lugar de `1250000`: el guion bajo se ignora y los montos grandes se leen mucho mejor.

In [1]:
# Variables: el nombre va a la izquierda, el valor a la derecha. Python deduce el tipo.
ticker = "YPFD"
precio_cierre = 42.75
cantidad_acciones = 300
paga_dividendo = True

valor_posicion = precio_cierre * cantidad_acciones

print(ticker, "es de tipo:", type(ticker))
print(valor_posicion, "es de tipo:", type(valor_posicion))
print(cantidad_acciones, "es de tipo:", type(cantidad_acciones))
print(paga_dividendo, "es de tipo:", type(paga_dividendo))

# f-string: el prefijo f permite meter variables y formato dentro del texto
print(f"\nPosición en {ticker}: $U {valor_posicion:,.2f}")
print(f"¿Paga dividendo?: {paga_dividendo}")

YPFD es de tipo: <class 'str'>
12825.0 es de tipo: <class 'float'>
300 es de tipo: <class 'int'>
True es de tipo: <class 'bool'>

Posición en YPFD: $U 12,825.00
¿Paga dividendo?: True


In [9]:
# Listas: colección ordenada, se accede por posición. El primer elemento es el 0.
tickers = ["YPFD", "GGAL", "PAMP", "TXAR"]
precios = [42.75, 38.10, 12.60, 55.40]

print("Cantidad de posiciones :", len(tickers))

#Acceder a elementos de la lista (de a uno)
print("Primera y última       :", tickers[0], "y", tickers[-1])

#Acceder a elementos de la lista (multiples)
print("Las dos primeras       :", tickers[:2])

# Las listas se modifican en el lugar 
# Agregar un elemento al final
tickers.append("EDN")
precios.append(9.85)

#modificar un elemento
precios[1] = 39.10
print("Tickers                :", tickers)
print("Precios                :", precios)

# Operaciones (promedio y ordenamiento)
print("Precio promedio        :", sum(precios) / len(precios))
print("Precios ordenados      :", sorted(precios))

Cantidad de posiciones : 4
Primera y última       : YPFD y TXAR
Las dos primeras       : ['YPFD', 'GGAL']
Tickers                : ['YPFD', 'GGAL', 'PAMP', 'TXAR', 'EDN']
Precios                : [42.75, 39.1, 12.6, 55.4, 9.85]
Precio promedio        : 31.939999999999998
Precios ordenados      : [9.85, 12.6, 39.1, 42.75, 55.4]


In [11]:
# Diccionarios: colección de pares clave-valor. Es la forma natural de representar una fila.
gasto = {
    "descripcion": "Alquiler",
    "monto": -14_500,
    "categoria": "Vivienda",
}

print(gasto["descripcion"])
print(gasto["monto"])

# .get() devuelve un valor por omisión si la clave no está, en lugar de romper
print(gasto.get("medio_pago", "sin dato")) #Sirve para buscar elementos sin errores que paren la ejecución del programa

# Se agregan claves asignando
gasto["medio_pago"] = "Transferencia"
print(list(gasto.keys()))
print(gasto)

Alquiler
-14500
sin dato
['descripcion', 'monto', 'categoria', 'medio_pago']
{'descripcion': 'Alquiler', 'monto': -14500, 'categoria': 'Vivienda', 'medio_pago': 'Transferencia'}


In [14]:
# La estructura que vamos a usar todo el curso: una LISTA de DICCIONARIOS.
# Cada diccionario es una transacción, la lista es el extracto del mes.
movimientos = [
    {"descripcion": "Salario", "monto": 52_000, "categoria": "Ingresos"},
    {"descripcion": "Freelance", "monto": 8_000, "categoria": "Ingresos"},
    {"descripcion": "Alquiler", "monto": -14_500, "categoria": "Vivienda"},
    {"descripcion": "Cuota universidad", "monto": -9_800, "categoria": "Educacion"},
    {"descripcion": "Supermercado", "monto": -6_200, "categoria": "Supermercado"},
    {"descripcion": "Cuota mutualista", "monto": -4_200, "categoria": "Salud"},
    {"descripcion": "Combustible", "monto": -3_100, "categoria": "Transporte"},
    {"descripcion": "Restaurante", "monto": -1_850, "categoria": "Ocio"},
    {"descripcion": "Netflix", "monto": -390, "categoria": "Ocio"},
]

# Bucle con if: recorrer la lista y quedarse con lo que cumple una condición
total_ingresos = 0
total_egresos = 0
for movimiento in movimientos:
    if movimiento["monto"] > 0:
        total_ingresos += movimiento["monto"]
    else:
        total_egresos += movimiento["monto"]

print(f"Ingresos del mes: ${total_ingresos:,.0f}")
print(f"Egresos del mes: ${abs(total_egresos):,.0f}")
print(f"Balance del mes: ${total_ingresos + total_egresos:,.0f}")

Ingresos del mes: $60,000
Egresos del mes: $40,040
Balance del mes: $19,960


In [17]:
for movimiento in movimientos:
    if movimiento["monto"] < -5000:
        print(f"Gasto grande: {movimiento['descripcion']} ${movimiento['monto']:,.0f}")
    elif movimiento["monto"] < 0:
        print(f"Otros gastos: {movimiento['descripcion']} ${movimiento['monto']:,.0f}")
    else:
        print(f"Ingreso: {movimiento['descripcion']} ${movimiento['monto']:,.0f}")

Ingreso: Salario $52,000
Ingreso: Freelance $8,000
Gasto grande: Alquiler $-14,500
Gasto grande: Cuota universidad $-9,800
Gasto grande: Supermercado $-6,200
Otros gastos: Cuota mutualista $-4,200
Otros gastos: Combustible $-3,100
Otros gastos: Restaurante $-1,850
Otros gastos: Netflix $-390


**Interpretación económica.** El mes cerró con un excedente de $U 19.960 sobre ingresos de
$U 60.000, o sea una **tasa de ahorro** del 33,3% (la tasa de ahorro es el excedente dividido
los ingresos). Eso es alto para un presupuesto de hogar, y la razón está en los tres gastos
grandes: alquiler, universidad y supermercado explican $U 30.500 de los $U 40.040 de egresos.
Toda la sensibilidad del presupuesto está en esas tres líneas; recortar Netflix, que son $U 390,
no mueve la aguja.

Ese es el tipo de conclusion es la que se va a pedir siempre, no se busca que me muestren el numero solo sin ninguna conclusion

<h2 style='font-weight:bold; color:#003366;'>1. De la disciplina a la automatización</h2>

En el Tema 1 vimos los principios: nombres que revelan la intención, funciones cortas,
constantes con nombre en lugar de números mágicos, separar el cálculo de la impresión.

El problema de los principios es que dependen de que uno **se acuerde** de aplicarlos. 

El **linting** es un corrector ortográfico para el código. Un *linter* es un programa que lee tu archivo **sin
ejecutarlo** y te dice qué está mal: una variable que definiste y no usás, un import que sobró,
un nombre mal escrito, una línea que se fue de largo. 

Eso se llama **análisis estático**:
estático porque no hace falta correr el programa.

Vamos a usar tres herramientas, y la clave es que hacen tres trabajos distintos:

| Herramienta | Qué hace | ¿Modifica tu archivo? |
|---|---|---|
| **flake8** | Encuentra problemas de estilo y errores reales | No, solo informa |
| **black** | Reescribe el formato para que quede uniforme | Sí |
| **pylint** | Opina sobre el diseño y te da un puntaje sobre 10 | No, solo informa |

Para realizar instalaciones de librerias en Python:

- El **signo de exclamacion ! para Anaconda o % porcentaje para VSCode** al principio de una línea significa "esto no es Python, es un
  comando de la terminal". Así corremos las herramientas desde el notebook.
- Escribimos `python -m flake8` y no solo `flake8`. 

Las dos formas hacen lo mismo, pero la
  primera funciona siempre, incluso cuando la instalación quedó en una carpeta que la terminal
  no tiene en su camino de búsqueda. En Colab es el problema más frecuente y esta es la manera
  de esquivarlo.

In [1]:
VSCODE = True

if VSCODE:
    %pip install -q flake8 black pylint
else:
    # Si estás en Colab, esta celda instala las herramientas. En local ya las tenés.
    from importlib.util import find_spec

    HERRAMIENTAS = ["flake8", "black", "pylint"]
    faltantes = [nombre for nombre in HERRAMIENTAS if find_spec(nombre) is None]

    if faltantes:
        paquetes = " ".join(faltantes)
        print("Instalando:", paquetes)
        !pip install -q {paquetes}
    else:
        print("flake8, black y pylint ya están instaladas.")

# Otra forma de hacerlo, que funciona en Colab y en local, es usar el módulo de Python que instala paquetes:
"""
if  VSCODE:
    %python -m flake8 --version
else:
    !python -m flake8 --version
"""


Note: you may need to restart the kernel to use updated packages.


'\nif  VSCODE:\n    %python -m flake8 --version\nelse:\n    !python -m flake8 --version\n'

<h2 style='font-weight:bold; color:#003366;'>2. PEP 8, lo que importa de verdad</h2>

**PEP 8** es el documento oficial de estilo de Python. Tiene decenas de reglas y no hace falta
leerlo: las herramientas lo saben de memoria. Lo que sí conviene tener en la cabeza son las
seis reglas que aparecen el 95% de las veces.

1. **Nombres.** `snake_case` para variables y funciones (`tasa_anual`, `calcular_margen`).
   `MAYUSCULAS_CON_GUIONES` para constantes, o sea valores que no cambian durante la corrida
   (`TASA_IVA_BASICA = 0.22`).
2. **Imports arriba y agrupados**: primero la biblioteca estándar (`csv`, `pathlib`), después
   las librerías externas (`pandas`, `matplotlib`), después lo tuyo. Uno por línea.
3. **Espacios alrededor de los operadores.** `margen = ventas - costo`, no
   `margen=ventas-costo`.
4. **Dos líneas en blanco entre funciones** de nivel superior. Es lo que te permite ver de un
   scroll dónde termina una y empieza la otra.

### Esto sí / esto no

| Regla | Esto sí | Esto no |
|---|---|---|
| Variables y funciones | `tasa_anual`, `calcular_margen()` | `TasaAnual`, `CalcularMargen()`, `tasaanual` |
| Constantes | `TASA_IVA_BASICA = 0.22` | `tasa = 0.22` perdida en el medio del código |
| Imports | uno por línea, arriba, agrupados | `import os, csv, pandas` en el medio del archivo |
| Espacios | `margen = ventas - costo` | `margen=ventas-costo` |
| Paréntesis | `calcular_margen(ventas, costo)` | `calcular_margen( ventas,costo )` |
| Separación | dos líneas en blanco entre funciones | ninguna, o cinco |

<h2 style='font-weight:bold; color:#003366;'>3. flake8: el que encuentra errores</h2>

`flake8` lee el archivo y devuelve **una línea por problema**, con este formato:

```
archivo.py:LÍNEA:COLUMNA: CÓDIGO mensaje
```

El código tiene una letra que dice de qué familia es, y esa letra es lo que hay que mirar
primero:

- **`E` y `W`** (*error* y *warning* de estilo): cosas de formato. `E501` línea larga,
  `E225` falta un espacio. Molestas, cosméticas, y las arregla `black` sola.
- **`F`** (de *pyflakes*): **estos son bugs de verdad**. `F401` import sin usar, `F841`
  variable asignada y nunca usada, `F821` nombre no definido.
- **`C`**: complejidad excesiva de una función.

**Prioridad:** corregí primero todos los `F`. Los `E` y `W` los arregla `black`.

Para probarlo necesitamos un archivo `.py`. La magia `%%writefile` guarda el contenido de la
celda como archivo, que es la forma habitual de trabajar con archivos de código desde un
notebook.

Para ver mejor los errores podemos pedirle a jupyter que nos muestre los **numeros de linea** en ... > Line numbers o View > Line Numbers

In [3]:
%%writefile margen_sucio.py
import os
import csv

TASA_IVA=0.22

def calcular_margen_porcentual(ventas, costo):
    margen_bruto = ventas - costo
    margen_porcentual = margen_bruto / ventas * 100
    nota_para_el_analista = "Revisar con el area comercial antes de publicar el reporte mensual."
    return margen_porcentaul

print(calcular_margen_porcentual(1_250_000, 890_000))

Overwriting margen_sucio.py


> Ese archivo tiene 12 líneas y calcula el margen bruto porcentual de un negocio.
>
> **¿Cuántos problemas distintos les parece que va a reportar `flake8`?**
>
> Y la segunda pregunta, más importante: **¿alguno de esos problemas haría que el programa se
> rompa si lo ejecutáramos?**

In [4]:
!python -m flake8 --max-line-length=88 margen_sucio.py

margen_sucio.py:1:1: F401 'os' imported but unused
margen_sucio.py:2:1: F401 'csv' imported but unused
margen_sucio.py:4:9: E225 missing whitespace around operator
margen_sucio.py:6:1: E302 expected 2 blank lines, found 1
margen_sucio.py:8:5: F841 local variable 'margen_porcentual' is assigned to but never used
margen_sucio.py:9:5: F841 local variable 'nota_para_el_analista' is assigned to but never used
margen_sucio.py:9:89: E501 line too long (97 > 88 characters)
margen_sucio.py:10:12: F821 undefined name 'margen_porcentaul'
margen_sucio.py:12:1: E305 expected 2 blank lines after class or function definition, found 1


Nueve problemas. Vamos uno por uno, porque leer esta salida es exactamente la
habilidad que queremos:

| Código | Qué dice | Gravedad |
|---|---|---|
| `F401 'os' imported but unused` | Importaste `os` y no lo usás. Idem `csv`. | Código muerto: confunde al que lee |
| `E225 missing whitespace around operator` | `TASA_IVA=0.22` sin espacios alrededor del `=` | Cosmético |
| `E302 expected 2 blank lines, found 1` | Falta una línea en blanco antes del `def` | Cosmético |
| `F841 local variable ... assigned but never used` | `margen_porcentual` y `nota_para_el_analista` se calculan y no se usan | Sospechoso: o sobra, o falta usarlo |
| `E501 line too long (97 > 88 characters)` | La línea de la nota se pasó del límite | Cosmético |
| `F821 undefined name 'margen_porcentaul'` | **Este es el importante** | Bug fatal |
| `E305 expected 2 blank lines after ...` | Falta separación después de la función | Cosmético |

Detengámonos en `F821`. En el `return` dice `margen_porcentaul` en lugar de
`margen_porcentual`: dos letras cambiadas de lugar. Ese archivo **no puede funcionar**. Si lo
ejecutaras, revienta con `NameError` en el momento exacto en que alguien llame a la función,
que en un pipeline de datos puede ser media hora después de arrancar el proceso, o el martes
que viene cuando esa rama del `if` finalmente se cumpla.

`flake8` lo encontró en 0,2 segundos **sin ejecutar una sola línea**. Eso es el análisis
estático, y es el argumento entero de esta clase.

Fijate también en el `F841` de `margen_porcentual`: la variable que se calcula y nunca se usa
es la *pista* de que el `return` está mal. Los dos mensajes cuentan la misma historia.

Y fijate en lo que flake8 **no** dice: no se queja de que la función podría llamarse mejor, ni
de que falta un docstring. Eso es territorio de `pylint`.

> **Cambien el valor de `LARGO_MAXIMO` a 120 y vuelvan a correr la celda.**
>
> ¿Cuántos problemas reporta ahora `flake8`? ¿Cuál desapareció?
> **¿alguno de los problemas graves cambió al mover el límite?**

In [ ]:
LARGO_MAXIMO = 88

!python -m flake8 --max-line-length={LARGO_MAXIMO} margen_sucio.py

<h2 style='font-weight:bold; color:#003366;'>4. black: el que formatea</h2>

`black` se presenta a sí mismo como *the uncompromising code formatter*, el formateador que no
negocia. Reescribe el archivo aplicando un único estilo posible y casi no ofrece opciones para
discutirlo.

Eso, que suena rígido, es exactamente su valor: **elimina toda decisión de formato**. Tres
personas con tres criterios producen tres archivos idénticos, y las discusiones de "¿va espacio
acá?" desaparecen del equipo.

Qué hace concretamente: normaliza la indentación a 4 espacios, pone espacios alrededor de los
operadores y después de las comas, unifica las comillas en dobles, agrega las dos líneas en
blanco entre funciones y parte las líneas largas de código.

La opción `--diff` muestra los cambios que *haría*, sin tocar nada. Muy recomendable la primera
vez que lo corrés sobre algo que te importa.

In [6]:
!python -m black --diff --line-length 88 margen_sucio.py

--- margen_sucio.py	2026-08-19 19:17:47.909557+00:00
+++ margen_sucio.py	2026-08-19 19:22:02.412553+00:00
@@ -1,12 +1,16 @@
 import os
 import csv
 
-TASA_IVA=0.22
+TASA_IVA = 0.22
+
 
 def calcular_margen_porcentual(ventas, costo):
     margen_bruto = ventas - costo
     margen_porcentual = margen_bruto / ventas * 100
-    nota_para_el_analista = "Revisar con el area comercial antes de publicar el reporte mensual."
+    nota_para_el_analista = (
+        "Revisar con el area comercial antes de publicar el reporte mensual."
+    )
     return margen_porcentaul
 
+
 print(calcular_margen_porcentual(1_250_000, 890_000))


would reformat margen_sucio.py

All done! ✨ 🍰 ✨
1 file would be reformatted.


In [8]:
# Ahora sí, lo aplicamos, miramos el archivo resultante y volvemos a correr flake8
!python -m black -q --line-length 88 margen_sucio.py
print("**margen_sucio.py** después de black")

print("\nLo que flake8 sigue reportando")
!python -m flake8 --max-line-length=88 margen_sucio.py

**margen_sucio.py** después de black

Lo que flake8 sigue reportando
margen_sucio.py:1:1: F401 'os' imported but unused
margen_sucio.py:2:1: F401 'csv' imported but unused
margen_sucio.py:9:5: F841 local variable 'margen_porcentual' is assigned to but never used
margen_sucio.py:10:5: F841 local variable 'nota_para_el_analista' is assigned to but never used
margen_sucio.py:13:12: F821 undefined name 'margen_porcentaul'


Pasamos de **9 problemas a 5**, y los que quedan son precisamente los interesantes:

- `F401` los dos imports sin usar,
- `F841` las dos variables muertas,
- `F821` el nombre mal escrito.

Todos los `E` desaparecieron solos. Incluso el `E501`: `black` no cortó el texto de la nota, lo
metió entre paréntesis y lo bajó a una línea propia, que ahora entra en 88 caracteres.

Esto ilustra la división de tareas mejor que cualquier explicación. **Lo que `black` arregló era
formato. Lo que queda requiere una decisión humana**, porque borrar código es una decisión: hay
que mirar si `import csv` sobra o si falta la línea que lo usa.

Qué **no** hace `black`, y conviene decirlo explícitamente porque genera confusión:

- **No borra imports sin usar.** Nunca elimina código.
- **No renombra nada.** Si tu función se llama `f`, sigue llamándose `f`.
- **No corrige lógica.** El `return margen_porcentaul` sigue roto después de formatear, y ese es
  el único problema del archivo que impide que funcione.
- **No parte el contenido de un texto.** Puede reacomodarlo, como acabamos de ver, pero si el
  string por sí solo mide más de 88 caracteres el `E501` se queda: cortarlo cambiaría el dato.

**Formato y estilo son dos cosas distintas.** El formato es mecánico y se automatiza al 100%.
El estilo (nombres, estructura, qué borrar) es criterio, y el criterio no se delega.

<h2 style='font-weight:bold; color:#003366;'>5. pylint: el que opina sobre el diseño</h2>

`pylint` va bastante más allá del formato: mira el **diseño** del código y termina dándote un
**puntaje de 0 a 10**.

Sus códigos también empiezan con una letra:

- **`C`** convención: nombres, docstrings faltantes.
- **`R`** refactorización sugerida: hay una forma más simple de escribir esto.
- **`W`** advertencia: algo raro pero no fatal.
- **`E`** error probable: variable que no existe, argumentos que no coinciden.

### Los mensajes que valen la pena

De todo lo que dice `pylint`, estos cuatro son los que casi siempre conviene atender:

| Mensaje | Qué te está diciendo | Por qué importa |
|---|---|---|
| `invalid-name` (C0103) | El nombre no sigue la convención: `CalcularMargen` en lugar de `calcular_margen` | Es lo único que ninguna otra herramienta te avisa |
| `missing-function-docstring` (C0116) | La función no tiene docstring | Vale la pena **cuando el nombre no alcanza**. Si alcanza, se silencia |
| `too-many-arguments` (R0913) | La función recibe demasiados parámetros | Casi siempre significa que hace más de una cosa |
| `duplicate-code` (R0801) | El mismo bloque aparece copiado en dos lugares | Es el aviso de que falta una función |

### Los que se pueden silenciar sin culpa

Y estos, en análisis de datos, molestan más de lo que ayudan:

- `invalid-name` cuando el nombre es una convención universal: `ax` y `fig` en matplotlib, `df`
  para un DataFrame de trabajo en una celda exploratoria.
- `too-few-public-methods`, que asume que estás escribiendo clases.
- `missing-module-docstring` en un script de tres líneas.

Se silencian con un comentario `# pylint: disable=nombre-del-mensaje`, que se puede poner en
una línea suelta o al principio de una función.

**Regla de convivencia: silenciar una regla exige una justificación.** Si la razón es "me daba
error", la solución correcta era arreglar el código.

Corrámoslo sobre el mismo archivo que ya pasó por `black`.

In [9]:
!python -m pylint margen_sucio.py

************* Module margen_sucio
margen_sucio.py:1:0: C0114: Missing module docstring (missing-module-docstring)
margen_sucio.py:7:0: C0116: Missing function or method docstring (missing-function-docstring)
margen_sucio.py:13:11: E0602: Undefined variable 'margen_porcentaul' (undefined-variable)
margen_sucio.py:9:4: W0612: Unused variable 'margen_porcentual' (unused-variable)
margen_sucio.py:10:4: W0612: Unused variable 'nota_para_el_analista' (unused-variable)
margen_sucio.py:1:0: W0611: Unused import os (unused-import)
margen_sucio.py:2:0: W0611: Unused import csv (unused-import)

-----------------------------------
Your code has been rated at 0.00/10



**0,00 sobre 10.** Fijate que `pylint` encuentra el mismo error que `flake8`
(`E0602: Undefined variable`, que es el `F821` con otro nombre) y le agrega dos cosas que
`flake8` no mira: que falta el docstring del módulo y que falta el de la función.

Ahora veamos el otro lado: cómo se silencia un mensaje cuando de verdad no aplica.

In [10]:
%%writefile reporte_tablero.py
"""Configuración de los filtros del tablero comercial."""


def armar_filtros(anio, mes, region, canal, categoria, moneda):
    # pylint: disable=too-many-arguments, too-many-positional-arguments
    """Junta los filtros del tablero. Son seis porque el tablero tiene seis filtros."""
    return {
        "anio": anio,
        "mes": mes,
        "region": region,
        "canal": canal,
        "categoria": categoria,
        "moneda": moneda,
    }


print(armar_filtros(2026, 6, "Montevideo", "Web", "Hogar", "UYU"))

Writing reporte_tablero.py


In [11]:
# Con el disable puesto y justificado, pylint no protesta
!python -m pylint reporte_tablero.py


------------------------------------
Your code has been rated at 10.00/10



<h2 style='font-weight:bold; color:#003366;'>6. Una nota sobre las anotaciones de tipo</h2>

Si buscás código de Python en internet vas a ver firmas de función con **anotaciones de tipo**
(*type hints*). Existen, se ven así:

```python
def clasificar_mora(dias_atraso: int, moneda: str):
    ...
```

Después de cada parámetro van dos puntos y el tipo que se espera. También se puede declarar qué
devuelve la función, agregando al final de la firma una flecha `->` con el tipo, justo antes de
los dos puntos.

Tres cosas que conviene saber:

1. Python **no las verifica**. Si la anotación dice `int` y le pasás texto, Python no dice nada.
   No son una validación, son documentación.
2. Sirven de verdad en **equipos de desarrollo grandes**, donde una función la escribe una
   persona y la llama otra que nunca vio el código, y donde el editor usa las anotaciones para
   autocompletar y avisar de incompatibilidades.
3. **En este curso no las vamos a usar.** En análisis de datos agregan ruido: nuestras funciones
   reciben listas de diccionarios y DataFrames, los tipos reales son largos de escribir, y el
   nombre del parámetro más una línea de docstring ya dicen todo lo que hace falta saber.

Lo esperado para el curso es: **firma sin anotaciones, docstring de una
línea, y dos o tres líneas solo cuando hay un supuesto de negocio que el código no puede
expresar.**

<h2 style='font-weight:bold; color:#003366;'>7. Cómo se usa esto en la práctica</h2>

Correr los comandos a mano funciona para aprender, pero lo que cambia el hábito es que las
herramientas actúen solas.

**En VS Code.** Instalá las extensiones oficiales de Python y Black Formatter, y agregá esto en
`.vscode/settings.json` dentro de la carpeta del proyecto:

```json
{
  "editor.formatOnSave": true,
  "editor.defaultFormatter": "ms-python.black-formatter"
}
```

A partir de ahí, cada vez que guardás el archivo queda formateado, y flake8 te subraya los
problemas en el margen mientras escribís. Es el cambio de hábito más rentable del curso: cinco
minutos de configuración una vez.

**En Colab.** No hay integración automática, así que el patrón es el que usamos hoy: escribir el
código en una celda con `%%writefile`, y correr las herramientas con `!python -m ...`.

### El flujo de tres pasos

Este es el flujo que queremos que repitan, en este orden, y no es negociable el orden:

1. **`black archivo.py`**: arregla todo el formato de una. Gratis, sin pensar.
2. **`flake8 archivo.py`**: corregí a mano lo que quedó, empezando por los `F`.
3. **`pylint archivo.py`**: mirá el puntaje y decidí, mensaje por mensaje, qué corregís y qué
   silenciás con justificación.

Primero `black`, porque si corrés `flake8` antes te vas a pasar diez minutos arreglando espacios
que la máquina arreglaba sola.

> ### Ejercicio en clase
>
> Abajo está el peor fragmento de código de todo el notebook. Es un análisis de cartera de
> créditos: nombres de una letra, imports que sobran, un
> `for` con `range(len(...))`, un `print` adentro de la función, y un nombre mal escrito.
>
> **El número: ¿cuánto saca en `pylint`, y a cuánto lo podemos llevar entre todos en cinco
> minutos?**


In [12]:
%%writefile cartera_peor.py
import os
import csv

def AnalizarCartera( l,U ):
    T=0
    for i in range(len(l)):
        T=T+l[i]
    prom = T/len(l)
    mora=[]
    for i in range(len(l)):
        if l[i]>U:
            mora.append(l[i])
    sin_usar=0
    print("Promedio de la cartera:",prom)
    print("Creditos sobre el umbral:",len(mora))
    return prom,mroa

AnalizarCartera([1200,4500,3000,8900,2100],2500)

Writing cartera_peor.py


In [13]:
!python -m flake8 --max-line-length=88 cartera_peor.py
print()
!python -m pylint cartera_peor.py

cartera_peor.py:1:1: F401 'os' imported but unused
cartera_peor.py:2:1: F401 'csv' imported but unused
cartera_peor.py:4:1: E302 expected 2 blank lines, found 1
cartera_peor.py:4:21: E201 whitespace after '('
cartera_peor.py:4:22: E741 ambiguous variable name 'l'
cartera_peor.py:4:23: E231 missing whitespace after ','
cartera_peor.py:4:25: E202 whitespace before ')'
cartera_peor.py:5:6: E225 missing whitespace around operator
cartera_peor.py:7:10: E225 missing whitespace around operator
cartera_peor.py:9:9: E225 missing whitespace around operator
cartera_peor.py:11:16: E225 missing whitespace around operator
cartera_peor.py:13:5: F841 local variable 'sin_usar' is assigned to but never used
cartera_peor.py:13:13: E225 missing whitespace around operator
cartera_peor.py:14:36: E231 missing whitespace after ','
cartera_peor.py:15:38: E231 missing whitespace after ','
cartera_peor.py:16:16: E231 missing whitespace after ','
cartera_peor.py:16:17: F821 undefined name 'mroa'
cartera_peor.py:1

In [ ]:
%%writefile cartera_corregida.py

## Corregí los errores de estilo y de sintaxis que flake8 y pylint detectaron en cartera_peor.py
import os
import csv

def AnalizarCartera( l,U ):
    T=0
    for i in range(len(l)):
        T=T+l[i]
    prom = T/len(l)
    mora=[]
    for i in range(len(l)):
        if l[i]>U:
            mora.append(l[i])
    sin_usar=0
    print("Promedio de la cartera:",prom)
    print("Creditos sobre el umbral:",len(mora))
    return prom, mroa

AnalizarCartera([1200,4500,3000,8900,2100],2500)

In [ ]:
!python -m black -q --line-length 88 cartera_corregida.py
!python -m flake8 --max-line-length=88 cartera_corregida.py
print("flake8: sin observaciones\n")
!python -m pylint cartera_corregida.py
print()
!python cartera_corregida.py

<h2 style='font-weight:bold; color:#003366; text-align:center;'>Ejercicios para resolver en clase</h2>

La cantidad de asteriscos indica la dificultad. Para los tres, el procedimiento es el
mismo y es el flujo de tres pasos:

1. Corré las herramientas sobre el archivo tal como está y anotá qué reportan.
2. Reescribí el archivo en la celda vacía que sigue, con `%%writefile`, hasta que `flake8` no
   diga nada y `pylint` dé 8 o más.
3. Recién después desplegá la solución para comparar los enfoques.

Los fragmentos son cortos a propósito: son 8 o 10 líneas cada uno, el trabajo está en las
decisiones, no en la cantidad de tipeo.

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 1 *</h3>

Este archivo calcula el valor futuro de un plazo fijo. Funciona, pero los
nombres no dicen nada y hay un error de estilo que `black` arregla solo.

Corré el flujo de tres pasos y dejalo con `flake8` limpio y `pylint` en 8 o más. Después
escribí en un comentario **qué calcula la función**, ahora que se puede saber sin leer el
cuerpo.

```python
%%writefile ejercicio_1.py
def calc(m,t,n):
    r=m # capital inicial
    r=r * (1 + t) ** n #valor futuro del capital
    g=r-m #Interes ganado
    return r,g
print(calc(50000,0.045,5))
```

In [ ]:
# Escribí acá tu versión. Ojo: la magia %%writefile tiene que ser la PRIMERA
# línea de la celda, así que borrá estos comentarios antes de correrla.
#
# %%writefile ejercicio_1.py


<details>
<summary><b>Ver solución</b></summary>

```python
%%writefile ejercicio_1.py
"""Valor futuro de un plazo fijo con capitalización anual."""


def calcular_valor_futuro(capital_inicial, tasa_anual, anios):
    return capital_inicial * (1 + tasa_anual) ** anios

capital = 50_000
valor_futuro = calcular_valor_futuro(capital, tasa_anual=0.045, anios=5)

print(f"Capital inicial : $U {capital:,.2f}")
print(f"Valor futuro    : $U {valor_futuro:,.2f}")
print(f"Interés ganado  : $U {valor_futuro - capital:,.2f}")


# !python -m black -q ejercicio_1.py
# !python -m flake8 ejercicio_1.py    -> sin salida
# !python -m pylint ejercicio_1.py    -> 9.00/10

# Qué calcula: el capital acumulado después de 5 años al 4,5% anual, y cuánto de ese
# total es interés. $U 50.000 se convierten en $U 62.309: el interés compuesto aportó
# $U 12.309, un 24,6% sobre el capital inicial.
#
# La función original devolvía una tupla de dos cosas; separar el cálculo del valor futuro
# de la resta del capital deja cada número donde corresponde.
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 2 *</h3>

Este archivo liquida un sueldo. El problema no es el estilo: son los
**números mágicos**. Cinco porcentajes sueltos que nadie puede auditar y que, cuando cambie la
franja del IRPF, alguien va a modificar en el lugar equivocado.

Dado que la tasa de aporte jubilatorio general es de 0.15, para los de sueldo menor a $90000 la tasa de descuento IRPF es de 0.08 y para los de sueldo mayor es 0.12

Reescribilo con constantes con nombre en MAYÚSCULAS y dejalo con `flake8` limpio y `pylint` en 8
o más. Después contestá: **¿qué pasa si el nominal es exactamente 90.000?**

```python
%%writefile ejercicio_2.py
def liq(n):
    a=n*0.15
    i=n*0.08
    if n>90000:
        i=n*0.12
    return n-a-i
print(liq(85000))
print(liq(120000))
```

In [ ]:
# Tu versión acá (arrancá con %%writefile ejercicio_2.py en la primera línea)


<details>
<summary><b>Ver solución</b></summary>

```python
%%writefile ejercicio_2.py
"""Liquidación de sueldo nominal a líquido según los tramos vigentes."""

TASA_APORTES_JUBILATORIOS = 0.15
TASA_IRPF_TRAMO_BAJO = 0.08
TASA_IRPF_TRAMO_ALTO = 0.12
TOPE_TRAMO_BAJO = 90_000


def calcular_tasa_irpf(sueldo_nominal):
    """Devuelve la tasa de IRPF del tramo que corresponde al nominal.

    El tope es inclusivo: un nominal de exactamente 90.000 tributa en el tramo bajo.
    """
    if sueldo_nominal > TOPE_TRAMO_BAJO:
        return TASA_IRPF_TRAMO_ALTO
    return TASA_IRPF_TRAMO_BAJO


def calcular_sueldo_liquido(sueldo_nominal):
    aportes = sueldo_nominal * TASA_APORTES_JUBILATORIOS
    irpf = sueldo_nominal * calcular_tasa_irpf(sueldo_nominal)
    return sueldo_nominal - aportes - irpf



"""Imprime la liquidación de los dos sueldos de ejemplo."""
for nominal in (85_000, 90_000, 120_000):
    liquido = calcular_sueldo_liquido(nominal)
    print(f"Nominal $U {nominal:>8,} -> líquido $U {liquido:>9,.0f} "
            f"({liquido / nominal:.1%} del nominal)")


# Qué pasa con 90.000 exactos: la condición es > y no >=, así que tributa en el tramo
# bajo. Eso NO se podía saber leyendo el original: era un supuesto escondido en un
# operador. Por eso la función que elige la tasa tiene docstring de dos líneas y el
# resto no: acá hay una decisión de negocio que el código solo no expresa.
#
# Interpretación: un nominal de 85.000 deja 77% en el bolsillo y uno de 120.000 deja
# 73%. El salto de tramo se come 4 puntos de sueldo líquido, y eso importa cuando se
# negocia un aumento cerca del tope.
```

</details>

<h3 style='font-weight:bold; color:#BB3300; text-align:left;'>EJERCICIO 3 **</h3>

Este archivo está **bien formateado** y `flake8` no reporta nada. `pylint`,
en cambio, tiene cuatro cosas para decir.

Corré `pylint`, leé los cuatro mensajes y decidí, **uno por uno**, si lo corregís o lo silenciás
con `# pylint: disable=...`. Lo que se pide no es el puntaje: es la **justificación escrita** de
cada decisión, en un comentario al lado.

```python
%%writefile ejercicio_4.py
import matplotlib.pyplot as plt


def graficar(df, ax, titulo, color, ancho, alto, grilla):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    ax.bar(df["categoria"], df["monto"], color=color)
    ax.set_title(titulo)
    if grilla:
        ax.grid(axis="y")
    return fig
```

In [17]:
%%writefile ejercicio_4.py
import matplotlib.pyplot as plt


def graficar(df, ax, titulo, color, ancho, alto, grilla):
    fig, ax = plt.subplots(figsize=(ancho, alto))
    ax.bar(df["categoria"], df["monto"], color=color)
    ax.set_title(titulo)
    if grilla:
        ax.grid(axis="y")
    return fig

Writing ejercicio_4.py


<details>
<summary style="cursor:pointer; color:#0033BB;"><b>Pista</b> (abrila solo después de intentarlo)</summary>

Los cuatro mensajes son `missing-module-docstring`,
`missing-function-docstring`, `too-many-arguments` y `too-many-positional-arguments`.

Pensalos así: los dos primeros son sobre documentación y hay que decidir en cada caso si el
docstring aporta algo o es ceremonia. Los dos últimos son en realidad **el mismo problema visto
dos veces**: la función recibe siete parámetros, y eso casi siempre significa que hace más de
una cosa.

Fijate también en lo que `pylint` **no** dice: no se queja de `df`, ni de `ax`, ni de `fig`,
porque acepta nombres cortos como variables locales. Y no se queja de que la función reciba `ax`
y lo sobreescriba en la primera línea, que es un bug de verdad. Ese lo tenés que ver vos: las
herramientas no reemplazan la lectura.

</details>